# 3장 2강: RNN과 LSTM — 실습문제 풀이

> **풀이 기준**: 업로드된 `jena_climate_2009_2016.csv`의 처음 12행으로 코드를 실제 실행한 결과입니다. 출력은 소수 여섯째 자리에서 반올림했습니다.
>
> **가정한 내용**: 별도 판단이 필요한 부분은 없었습니다. 제공 함수와 가중치를 그대로 사용했습니다.

---

## 실습 준비 (제공 코드)

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("jena_climate_2009_2016.csv", nrows=12)
times = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")
x = df["T (degC)"].iloc[:6].to_numpy(dtype=float) / 10.0
x_home = df["T (degC)"].iloc[6:12].to_numpy(dtype=float) / 10.0

print("읽은 데이터 크기:", df.shape)
print(df[["Date Time", "T (degC)"]].to_string(index=False))
print("선택 구간 간격(분):", times.diff().dropna().dt.total_seconds().unique() / 60)
print("선택 기온의 결측값 수:", df["T (degC)"].isna().sum())
print("실습 입력(기온/10):", x)
print("과제 입력(기온/10):", x_home)

읽은 데이터 크기: (12, 15)
          Date Time  T (degC)
01.01.2009 00:10:00     -8.02
01.01.2009 00:20:00     -8.41
01.01.2009 00:30:00     -8.51
01.01.2009 00:40:00     -8.31
01.01.2009 00:50:00     -8.27
01.01.2009 01:00:00     -8.05
01.01.2009 01:10:00     -7.62
01.01.2009 01:20:00     -7.62
01.01.2009 01:30:00     -7.91
01.01.2009 01:40:00     -8.43
01.01.2009 01:50:00     -8.76
01.01.2009 02:00:00     -8.88
선택 구간 간격(분): [10.]
선택 기온의 결측값 수: 0
실습 입력(기온/10): [-0.802 -0.841 -0.851 -0.831 -0.827 -0.805]
과제 입력(기온/10): [-0.762 -0.762 -0.791 -0.843 -0.876 -0.888]


In [2]:
def run_rnn(values, reset_each=False):
    h = 0.0
    rows = []
    for t, value in enumerate(values, start=1):
        h_prev = 0.0 if reset_each else h
        z = 0.5 * value + 0.5 * h_prev
        h = np.tanh(z)
        rows.append({"t": t, "x": value, "h_prev": h_prev, "z": z, "h": h})
    return pd.DataFrame(rows)


def sigmoid(value):
    return 1.0 / (1.0 + np.exp(-value))


def run_lstm(values):
    h = 0.0
    c = 0.0
    rows = []
    for t, value in enumerate(values, start=1):
        h_prev = h
        c_prev = c
        s = 0.5 * value + 0.2 * h_prev
        f = sigmoid(s + 1.0)
        i = sigmoid(s)
        g = np.tanh(s)
        o = sigmoid(s)
        keep = f * c_prev
        add = i * g
        c = keep + add
        h = o * np.tanh(c)
        rows.append({"t": t, "x": value, "h_prev": h_prev,
                     "f": f, "i": i, "g": g, "o": o,
                     "c_prev": c_prev, "keep": keep, "add": add,
                     "c": c, "h": h})
    return pd.DataFrame(rows)


print("RNN·LSTM 함수 준비 완료")

RNN·LSTM 함수 준비 완료


---

## 실습 1. 순차 데이터와 RNN의 동작 원리

### 문제 1-1. 이전 상태를 전달하면 무엇이 달라질까요?

In [3]:
rnn_table = run_rnn(x)
reset_table = run_rnn(x, reset_each=True)

print("[상태를 이어받는 RNN]")
print(rnn_table.round(6).to_string(index=False))
print()
print("[매 시점 초기화하는 RNN]")
print(reset_table.round(6).to_string(index=False))
print()
print("마지막 h (이어받음):", round(rnn_table["h"].iloc[-1], 6))
print("마지막 h (매번 초기화):", round(reset_table["h"].iloc[-1], 6))

[상태를 이어받는 RNN]
 t      x    h_prev         z         h
 1 -0.802  0.000000 -0.401000 -0.380804
 2 -0.841 -0.380804 -0.610902 -0.544762
 3 -0.851 -0.544762 -0.697881 -0.603021
 4 -0.831 -0.603021 -0.717010 -0.615054
 5 -0.827 -0.615054 -0.721027 -0.617545
 6 -0.805 -0.617545 -0.711273 -0.611474

[매 시점 초기화하는 RNN]
 t      x  h_prev       z         h
 1 -0.802     0.0 -0.4010 -0.380804
 2 -0.841     0.0 -0.4205 -0.397352
 3 -0.851     0.0 -0.4255 -0.401554
 4 -0.831     0.0 -0.4155 -0.393133
 5 -0.827     0.0 -0.4135 -0.391440
 6 -0.805     0.0 -0.4025 -0.382086

마지막 h (이어받음): -0.611474
마지막 h (매번 초기화): -0.382086


#### 답변

**수식 및 계산 과정**

- h₁ = tanh(0.5 × (−0.802) + 0.5 × 0) = tanh(−0.401) ≈ **−0.380804**
- h₂ = tanh(0.5 × (−0.841) + 0.5 × (−0.380804)) = tanh(−0.4205 − 0.190402) = tanh(−0.610902) ≈ **−0.544762**
- (비교) 매번 초기화한 h₂ = tanh(0.5 × (−0.841) + 0) = tanh(−0.4205) ≈ **−0.397352**

**Q1. 기온 데이터에서 시간 순서를 유지해야 하는 이유는 무엇인가요? 이번 입력 x와 원래 기온의 관계는 무엇인가요?**

- 같은 값들이라도 관측 순서에 따라 상승 흐름인지 하강 흐름인지가 달라집니다. RNN은 앞의 상태를 다음 시점에 넘기며 순서를 반영하므로, 순서를 섞으면 의미 있는 흐름이 사라집니다.
- x는 원래 기온(℃)을 **10으로 나눈 값**입니다(예: −8.02℃ → −0.802). 계산을 읽기 쉽게 하려는 고정 배율 조정입니다.

**Q2. 상태를 이어받는 RNN의 h1과 h2는 각각 얼마인가요? 두 번째 계산에 첫 번째 입력의 정보가 어떻게 들어가나요?**

- h₁ ≈ **−0.380804**, h₂ ≈ **−0.544762**입니다.
- 첫 입력(−0.802)으로 만든 h₁이 두 번째 계산에서 **h_prev로 들어가** 0.5 × h₁ = −0.190402만큼 가중합에 더해집니다. 이렇게 첫 입력의 정보가 h₁을 통해 전달됩니다.

**Q3. 두 경우의 마지막 h는 각각 얼마인가요? 첫 번째 h는 같고 두 번째부터 달라지는 이유를 설명하세요.**

- 이어받음 **−0.611474**, 매번 초기화 **−0.382086**입니다.
- 첫 시점에는 두 경우 모두 이전 상태가 0이라 계산이 같습니다. 두 번째부터는 이어받는 경우 h_prev = h₁(음수)이 더해지지만, 초기화하는 경우는 h_prev = 0이라 현재 입력만 반영하므로 값이 달라집니다.

**Q4. h가 음수이면 잘못 계산한 것인가요? 마지막 h에 10을 곱하면 다음 기온 예측값이 되나요?**

- 아닙니다. 입력이 음수이고 Tanh의 출력 범위가 −1~1이므로 음수 h는 정상입니다.
- 되지 않습니다. h는 신경망의 **내부 상태**일 뿐입니다. 기온을 예측하려면 h를 출력층으로 연결하고 정답 데이터로 학습해야 합니다. 이번 가중치는 학습한 값도 아닙니다.

---

## 실습 2. LSTM과 장기 의존성

### 문제 2-1. LSTM은 기억을 어떻게 남기고 더할까요?

In [4]:
lstm_table = run_lstm(x)

print("[게이트 표]")
print(lstm_table[["t", "x", "f", "i", "g", "o"]].round(6).to_string(index=False))
print()
print("[상태 표]")
print(lstm_table[["t", "c_prev", "keep", "add", "c", "h"]].round(6).to_string(index=False))
print()
row2 = lstm_table.iloc[1]
print("두 번째 시점 keep + add =", round(row2["keep"] + row2["add"], 6), " c =", round(row2["c"], 6))
print("마지막 c:", round(lstm_table["c"].iloc[-1], 6), " 마지막 h:", round(lstm_table["h"].iloc[-1], 6))

[게이트 표]
 t      x        f        i         g        o
 1 -0.802 0.645427 0.401072 -0.380804 0.401072
 2 -0.841 0.638150 0.393492 -0.407539 0.393492
 3 -0.851 0.635213 0.390466 -0.418071 0.390466
 4 -0.831 0.636414 0.391702 -0.413781 0.391702
 5 -0.827 0.636164 0.391444 -0.414676 0.391444
 6 -0.805 0.638278 0.393625 -0.407075 0.393625

[상태 표]
 t    c_prev      keep       add         c         h
 1  0.000000  0.000000 -0.152730 -0.152730 -0.060784
 2 -0.152730 -0.097465 -0.160363 -0.257828 -0.099263
 3 -0.257828 -0.163776 -0.163243 -0.327019 -0.123325
 4 -0.327019 -0.208119 -0.162079 -0.370198 -0.138727
 5 -0.370198 -0.235507 -0.162323 -0.397829 -0.148001
 6 -0.397829 -0.253926 -0.160235 -0.414161 -0.154301

두 번째 시점 keep + add = -0.257828  c = -0.257828
마지막 c: -0.414161  마지막 h: -0.154301


#### 답변

**수식 및 계산 과정**

- 첫 시점: s₁ = 0.5 × (−0.802) + 0.2 × 0 = −0.401
  - f₁ = σ(0.599) ≈ 0.645427, i₁ = o₁ = σ(−0.401) ≈ 0.401072, g₁ = tanh(−0.401) ≈ −0.380804
  - keep = 0.645427 × 0 = 0, add = 0.401072 × (−0.380804) ≈ −0.152730
  - c₁ = 0 + (−0.152730) = **−0.152730**
  - h₁ = 0.401072 × tanh(−0.152730) ≈ 0.401072 × (−0.151553) ≈ **−0.060784**
- 두 번째 시점: keep = 0.638150 × (−0.152730) ≈ −0.097465, add ≈ −0.160363
  - c₂ = −0.097465 + (−0.160363) = **−0.257828** (코드의 c와 일치)

**Q1. 망각·입력·출력 게이트와 후보 g의 역할은 각각 무엇인가요? f=0.8은 이전 기억의 80%를 버린다는 뜻인가요?**

- 망각 게이트 f: 이전 셀 상태 c를 얼마나 **남길지** 정합니다.
- 입력 게이트 i: 새 후보 정보 g를 얼마나 **더할지** 정합니다.
- 후보 g: 현재 입력과 이전 h로 만든 새 기억 후보입니다(음수 가능).
- 출력 게이트 o: tanh(c)를 은닉 상태 h에 얼마나 **드러낼지** 정합니다.
- f = 0.8은 이전 기억의 80%를 **남긴다**(20%를 줄인다)는 뜻입니다. keep = f × c_prev이기 때문입니다.

**Q2. 첫 시점 c1과 h1은 각각 얼마인가요? 첫 시점의 keep이 0인 이유는 무엇인가요?**

- c₁ ≈ **−0.152730**, h₁ ≈ **−0.060784**입니다.
- 첫 입력 전 셀 상태 c₀가 0이므로 keep = f × 0 = 0입니다. 남길 이전 기억이 아직 없습니다.

**Q3. 두 번째 시점의 keep, add, c는 각각 얼마인가요? 마지막 c와 h도 쓰고, 둘이 같은 값이어야 하는지 설명하세요.**

- 두 번째 시점: keep ≈ **−0.097465**, add ≈ **−0.160363**, c ≈ **−0.257828**입니다.
- 마지막 c ≈ **−0.414161**, h ≈ **−0.154301**입니다.
- 같을 필요가 없습니다. h = o × tanh(c)이므로 c에 tanh를 적용하고 출력 게이트 비율(약 0.39)을 곱한 값이 h입니다. c는 기억을 이어가는 상태, h는 그중 밖으로 드러내는 상태입니다.

**Q4. 출력 게이트 o는 미래 기온을 직접 출력하는 층인가요? 이번 예시에서 i와 o가 같은 이유는 무엇인가요?**

- 아닙니다. o는 셀 상태를 은닉 상태 h로 얼마나 드러낼지 정하는 0~1 비율입니다. 기온을 예측하려면 h 뒤에 별도 출력층과 학습이 필요합니다.
- 이번 예시는 계산을 단순하게 하려고 i와 o 모두 같은 가중합 s에 편향 0의 Sigmoid를 적용했기 때문입니다(i = o = σ(s)). 일반적인 LSTM에서는 게이트마다 가중치·편향이 따로 학습되므로 값이 다릅니다.

---

### 문제 2-2. 멀리 떨어진 과거 기억을 유지하기

In [5]:
rows = []
for f in [0.5, 0.9]:
    c = 1.0
    for step in range(1, 13):
        c = f * c
        if step in [1, 6, 12]:
            rows.append({"f": f, "step": step, "c": c})
memory_table = pd.DataFrame(rows)
print(memory_table.round(6).to_string(index=False))

  f  step        c
0.5     1 0.500000
0.5     6 0.015625
0.5    12 0.000244
0.9     1 0.900000
0.9     6 0.531441
0.9    12 0.282430


#### 답변

**수식 및 계산 과정**

- f = 0.5: 0.5¹ = 0.5, 0.5⁶ = 0.015625, 0.5¹² ≈ 0.000244
- f = 0.9: 0.9¹ = 0.9, 0.9⁶ ≈ 0.531441, 0.9¹² ≈ 0.282430

**Q1. f=0.5와 f=0.9에서 1·6·12단계 후 남은 기억은 각각 얼마인가요? 어떤 경우에 더 많이 남나요?**

| 단계 | f = 0.5 | f = 0.9 |
|---:|---:|---:|
| 1 | 0.5 | 0.9 |
| 6 | 0.015625 | 0.531441 |
| 12 | 0.000244 | 0.282430 |

- **f = 0.9**일 때 훨씬 많이 남습니다. 12단계 후에도 처음 기억의 약 28%가 남지만, f = 0.5는 거의 0이 됩니다.

**Q2. 이 실험을 RNN과 LSTM의 예측 성능 비교라고 할 수 있나요? 0.5를 RNN, 0.9를 LSTM의 고정된 성질로 이해해도 되나요?**

- 할 수 없습니다. 셀 상태 식의 한 항(f × c)만 떼어 내고, 새 정보도 0으로 고정한 원리 확인용 실험입니다. 예측도 학습도 하지 않았습니다.
- 안 됩니다. 0.5와 0.9는 설명을 위해 정한 망각 게이트 값입니다. 실제 LSTM의 f는 입력과 이전 상태에 따라 매 시점 다르게 계산되고, 학습으로 조정됩니다.

**Q3. 장기 의존성 문제가 무엇이며 LSTM은 어떻게 완화하나요? LSTM이라면 언제나 모든 과거를 완벽히 기억하나요?**

- 장기 의존성 문제: 여러 시점 전의 중요한 정보가 현재 판단에 필요한데, 기본 RNN은 상태를 반복 변환하고 학습 때 기울기가 여러 단계를 거치며 소실·폭주하기 쉬워 긴 관계를 학습하기 어려운 문제입니다.
- LSTM은 셀 상태를 **덧셈 위주(keep + add)**로 이어가고, 망각 게이트가 1에 가까우면 정보를 오래 유지할 수 있어 이 문제를 완화합니다.
- 완벽하지는 않습니다. f가 1보다 작으면 기억은 점점 줄고(12단계 후 28%), 아주 긴 시퀀스나 데이터·학습이 부족한 경우에는 LSTM도 먼 과거를 잘 활용하지 못할 수 있습니다.

**Q4. 기본 RNN과 LSTM의 공통점·차이점을 정리하고, 활용 분야 두 가지를 순서 정보가 필요한 이유와 함께 설명하세요.**

- 공통점: 입력을 시간 순서대로 하나씩 처리하고, 이전 상태를 다음 시점으로 전달하며, 모든 시점에서 같은 가중치를 사용합니다.
- 차이점: 기본 RNN은 은닉 상태 h 하나를 tanh로 매번 덮어씁니다. LSTM은 h와 별도로 셀 상태 c를 두고, 망각·입력·출력 게이트로 기억의 유지·추가·노출을 조절해 긴 관계를 학습하기 더 쉽습니다.
- 활용 분야
  - **기상·전력 수요 예측**: 앞선 몇 시간의 기온·사용량 흐름(상승 중인지, 하강 중인지)이 다음 값에 영향을 줍니다.
  - **설비 이상 감지**: 진동·온도가 서서히 올라가는 순서와 추세가 고장 징후가 되므로, 값의 순서를 함께 봐야 합니다.

---

## 기초 과제. 다음 기온 구간에서 상태 전달 확인하기

### 문제 3-1. 같은 RNN·LSTM을 다른 구간에 적용하기

In [6]:
home_rnn = run_rnn(x_home)
home_lstm = run_lstm(x_home)

print("[과제 RNN]")
print(home_rnn[["t", "x", "h"]].round(6).to_string(index=False))
print()
print("[과제 LSTM]")
print(home_lstm[["t", "x", "c", "h"]].round(6).to_string(index=False))
print()
first = home_lstm.iloc[0]
print("첫 LSTM: i1 =", round(first["i"], 6), " g1 =", round(first["g"], 6), " c1 = i1*g1 =", round(first["i"] * first["g"], 6))
print("마지막 RNN h:", round(home_rnn["h"].iloc[-1], 6))
print("마지막 LSTM c:", round(home_lstm["c"].iloc[-1], 6), " h:", round(home_lstm["h"].iloc[-1], 6))

[과제 RNN]
 t      x         h
 1 -0.762 -0.363576
 2 -0.762 -0.510043
 3 -0.791 -0.572021
 4 -0.843 -0.609113
 5 -0.876 -0.630687
 6 -0.888 -0.640690

[과제 LSTM]
 t      x         c         h
 1 -0.762 -0.147570 -0.059466
 2 -0.762 -0.246189 -0.097262
 3 -0.791 -0.314284 -0.121040
 4 -0.843 -0.362934 -0.135773
 5 -0.876 -0.396398 -0.145380
 6 -0.888 -0.418414 -0.151864

첫 LSTM: i1 = 0.405886  g1 = -0.363576  c1 = i1*g1 = -0.14757
마지막 RNN h: -0.64069
마지막 LSTM c: -0.418414  h: -0.151864


#### 답변

**수식 및 계산 과정**

- 첫 RNN: h₁ = tanh(0.5 × (−0.762) + 0.5 × 0) = tanh(−0.381) ≈ **−0.363576**
- 첫 LSTM: s₁ = 0.5 × (−0.762) = −0.381, i₁ = σ(−0.381) ≈ 0.405886, g₁ = tanh(−0.381) ≈ −0.363576
  - c₁ = f₁ × 0 + i₁ × g₁ = 0.405886 × (−0.363576) ≈ **−0.147570**

**Q1. 첫 RNN h와 첫 LSTM c는 각각 얼마인가요?**

- RNN h₁ ≈ **−0.363576**, LSTM c₁ ≈ **−0.147570**입니다(위 대입 과정).

**Q2. 마지막 RNN h와 LSTM c·h는 각각 얼마인가요? 실습과 같은 함수인데 숫자가 달라지는 이유는 무엇인가요?**

- 마지막 RNN h ≈ **−0.640690**, LSTM c ≈ **−0.418414**, LSTM h ≈ **−0.151864**입니다.
- 함수와 가중치는 같지만 **입력 기온 구간이 다르기** 때문입니다. 과제 구간(−7.62 → −8.88℃)은 실습 구간보다 더 낮은 기온으로 내려가 입력이 더 큰 음수가 됩니다.

**Q3. 처음에 상태를 0으로 시작하는 것과 매 시점 0으로 초기화하는 것은 어떻게 다른가요?**

- 처음에 0으로 시작: 첫 입력 전 상태만 0이고, 이후에는 계산한 상태를 계속 이어받습니다. 과거 정보가 누적됩니다(정상적인 RNN 동작).
- 매 시점 초기화: 매번 이전 상태를 0으로 지우므로, 각 시점이 현재 입력만 보고 계산됩니다. 순서 정보를 전달하지 못합니다.
- 예: 과제의 첫 두 입력은 모두 −0.762이지만, 이어받는 RNN에서는 h₁ ≈ −0.363576, h₂ ≈ −0.510043으로 다릅니다. h₂에는 h₁이 함께 반영되었기 때문입니다.

**Q4. 마지막 h의 크기만 보고 RNN과 LSTM 중 예측 성능이 더 좋은 모델을 고를 수 있나요?**

- 고를 수 없습니다. h는 내부 상태일 뿐 예측값이 아니며, 두 모델은 계산 구조가 달라 h의 크기를 직접 비교하는 것은 의미가 없습니다. 성능을 비교하려면 두 모델에 출력층을 붙여 같은 데이터로 학습시킨 뒤, 같은 검증 데이터의 예측 오차를 비교해야 합니다.

---

## 실습 마무리

**1. 순차 데이터에서 순서가 중요한 이유와, RNN이 이전 h를 전달하는 것이 은닉층을 쌓거나 학습을 반복하는 것과 어떻게 다른지 설명하세요.**

- 같은 값이라도 순서에 따라 상승·하강 흐름이 달라지므로 순서가 정보가 됩니다.
- RNN의 상태 전달은 **같은 가중치로 같은 계산을 시간 방향으로 반복**하며 내부 상태를 넘기는 것입니다. 은닉층을 여러 개 쌓는 것(층마다 다른 가중치)이나, 가중치를 조정하는 학습 반복과는 다릅니다. 이번 실습은 가중치를 고정한 순전파였습니다.

**2. 첫 입력 −0.802와 둘째 입력 −0.841을 처리한 h1·h2를 계산하세요. 매 시점 초기화하는 경우의 h2는 왜 다른가요?**

- h₁ = tanh(−0.401) ≈ −0.380804
- h₂ = tanh(0.5 × (−0.841) + 0.5 × (−0.380804)) = tanh(−0.610902) ≈ −0.544762
- 매 시점 초기화하면 h_prev = 0이므로 h₂ = tanh(−0.4205) ≈ −0.397352입니다. 첫 입력의 정보(h₁)가 빠졌기 때문에 다릅니다.

**3. LSTM의 f·i·g·o, c·h는 각각 어떤 역할을 하나요? f=0.8일 때 무엇의 80%를 남기며, 출력 게이트 o는 최종 예측 출력층과 같은가요?**

- f: 이전 c를 남길 비율 / i: 후보 g를 더할 비율 / g: 새 기억 후보 / o: tanh(c)를 h로 드러낼 비율 / c: 기억을 이어가는 셀 상태 / h: 다음 시점과 다음 층에 전달하는 은닉 상태
- f = 0.8이면 **이전 셀 상태 c_prev의 80%**를 남깁니다.
- o는 최종 예측 출력층이 아닙니다. LSTM 내부의 비율 값이며, 예측은 h 뒤의 별도 출력층이 담당합니다.

**4. c0=0, f1≈0.645427, i1=o1≈0.401072, g1≈−0.380804일 때 keep·add·c1·h1을 계산하세요. 다음 시점에 전달되는 두 상태는 무엇인가요?**

- keep = 0.645427 × 0 = **0**
- add = 0.401072 × (−0.380804) ≈ **−0.152730**
- c₁ = 0 + (−0.152730) = **−0.152730**
- h₁ = 0.401072 × tanh(−0.152730) ≈ 0.401072 × (−0.151553) ≈ **−0.060784**
- 다음 시점에는 **c₁(셀 상태)**과 **h₁(은닉 상태)** 두 가지가 전달됩니다.

**5. 초기 c=1에서 f=0.5와 f=0.9의 12단계 결과는 각각 얼마인가요? 장기 의존성과의 관계 및 이 실험으로 판단할 수 없는 것을 설명하세요.**

- f = 0.5: 0.5¹² ≈ **0.000244**, f = 0.9: 0.9¹² ≈ **0.282430**
- 망각 게이트가 1에 가까울수록 오래된 기억이 덜 사라져 먼 과거 정보를 현재까지 유지할 수 있습니다. 이것이 LSTM이 장기 의존성을 완화하는 원리입니다.
- 이 실험으로는 실제 예측 성능, RNN 대비 우열, 학습 시 기울기 전달 효과, f가 실제로 어떤 값으로 학습되는지를 판단할 수 없습니다.

**6. 기본 RNN과 LSTM의 공통점·차이점, 활용 분야 두 가지를 설명하세요. 과제에서 처음 두 입력이 같은데 상태가 다른 이유와 h·c만으로 성능을 비교할 수 없는 이유도 설명하세요.**

- 공통점: 시간 순서대로 처리하며 이전 상태를 다음 시점에 넘기고, 모든 시점에서 가중치를 공유합니다.
- 차이점: LSTM은 셀 상태 c와 세 게이트로 기억을 조절해 기본 RNN보다 긴 관계를 다루기 쉽습니다.
- 활용 분야: 기상·전력 수요 예측, 설비 센서 이상 감지(값의 흐름과 순서가 판단 근거)
- 과제의 첫 두 입력은 모두 −0.762이지만, 두 번째 계산에는 첫 시점의 상태(h₁, LSTM은 c₁도)가 함께 들어가므로 상태가 달라집니다.
- h·c는 학습하지 않은 고정 가중치로 만든 내부 상태이며 예측값이 아닙니다. 성능은 출력층과 학습을 거친 뒤 같은 검증 데이터의 예측 오차로 비교해야 합니다.